## Preparation

In [ ]:
import os,csv,random
import pandas as pd
import numpy as np
import scanpy as sc
import math

from skimage import io, color
import torch

from scanpy import read_10x_h5
import SpaGCN as spg
import matplotlib.pyplot as plt
import json

from tqdm import tqdm
import pickle

In [ ]:
# === CONFIGURE ME ===
DATA_DIR = 'data'        # input data root (subdirectories per dataset live underneath)
OUTPUT_DIR = 'output_Maaike'    # where this notebook writes its outputs
REPO_ROOT = '.'                # any remaining absolute-path references resolve to here
# ====================


In [ ]:
import GalaxyPython as gx
gx.__version__

In [ ]:
weekpoint1 = "5 wk regression"
weekpoint2 = "2 wk regression"

# compare within pos/neg group
group = "DHB pos" 
#group = "9AA neg"

DataDir = "data/Maaike"

In [ ]:
MALDIdata1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIloc1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIdata2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 
MALDIloc2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 

In [ ]:
mz_wk5 = pd.DataFrame(list(MALDIdata1.columns[1:]), index = list(MALDIdata1.columns[1:])).astype('float')
mz_wk5 = mz_wk5.rename(columns = {0:"m/z"})

mz_wk2 = pd.DataFrame(list(MALDIdata2.columns[1:]), index = list(MALDIdata2.columns[1:])).astype('float')
mz_wk2 = mz_wk2.rename(columns = {0:"m/z"})

In [ ]:
# pandas.dataset.iloc(row, column) is used for retrive rows and columns from a dataset
MALDIdataAnn1 = sc.AnnData(X = MALDIdata1.iloc[:,1:], var = mz_wk5, obs = MALDIloc1) # number of rows does not match
MALDIdataAnn2 = sc.AnnData(X = MALDIdata2.iloc[:,1:], var = mz_wk2, obs = MALDIloc2)

In [ ]:
MALDIdataAnn1.obs = MALDIdataAnn1.obs.astype(int)
sc.pp.normalize_per_cell(MALDIdataAnn1)

MALDIdataAnn2.obs = MALDIdataAnn2.obs.astype(int)
sc.pp.normalize_per_cell(MALDIdataAnn2)

## MSIWarp

In [ ]:
import msiwarp as mx
from msiwarp.util.warp import to_mx_peaks
from msiwarp.util.warp import to_mz, to_height

In [ ]:
def MSIwarping (MALDIdataAnn1, MALDIdataAnn2):
    spectra = []
    mzs = np.array(MALDIdataAnn1.var["m/z"]).astype(np.float) # peak m/z values
    meanspectrum1 =  np.mean(MALDIdataAnn1.X, axis = 0)
    for i in range(0,MALDIdataAnn1.X.shape[0],100): # 100?
        hs = meanspectrum1
        spectra.append( [mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs, hs))])
    #
    hs = meanspectrum1 # peak heights
    s = [mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs, hs))]
    
    #
    mzs2 = np.array(MALDIdataAnn2.var["m/z"]).astype(np.float) # peak m/z values
    meanspectrum2 =  np.mean(MALDIdataAnn2.X, axis = 0)
    hs2 = meanspectrum2 # peak heights
    s2 = [mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs2, hs2))]
    #
    reference_spectrum =  s2
    ### 1.2 setup the node placement parameters ######
    # method 1
    node_mzs = [i for i in range(15,1065,50)]
    node_deltas = [i*0.01 for i in range(15,121,5)] # [0.5] * len(node_mzs)# slacks = node_deltas * n_steps
    n_steps = 30
    nodes = mx.initialize_nodes(node_mzs, node_deltas, n_steps)
    epsilon = 1 # peak matching threshold, relative to peak width
    #
    optimal_moves = mx.find_optimal_spectra_warpings(spectra, reference_spectrum, nodes, epsilon)
    spectra2 = [[mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs.astype(np.float), mzs))]]
    #spectra2 = [[mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs.astype(np.float), hs))]]
    #
    warped_spectra = [mx.warp_peaks(s_i, nodes, optimal_moves[0]) for s_i in spectra2]
    #
    newmz = to_mz(warped_spectra[0])
    oldmz = to_height(warped_spectra[0])
    return {"newmz":newmz, "oldmz":oldmz}

In [ ]:
node_mzs = [i for i in range(15,1065,50)]
# [0.5] * len(node_mzs)

In [ ]:
# [i for i in range(15,121,5)]

In [ ]:
warpresult = MSIwarping(MALDIdataAnn1, MALDIdataAnn2)

In [ ]:
newint = warpresult.get("oldmz")
newint ## mean 

In [ ]:
mean_int = np.mean(MALDIdataAnn1.X, axis = 0)
mean_int[40:50]

In [ ]:
orgmz = warpresult.get("oldmz")

In [ ]:
len(orgmz)

In [ ]:
newmz = warpresult.get("newmz")

In [ ]:
newmz[50:60]

In [ ]:
MALDIdataAnn1.var[90:100]

In [ ]:
newmz

### Anchor point 1

In [ ]:
orgmz[1420:1430]

In [ ]:
newmz[1420:1430]

In [ ]:
newmz[1420:1430] #

In [ ]:
newmz

In [ ]:
MALDIdataAnn1.var.iloc[2075:2090]

### Anchor point 2

In [ ]:
orgmz[2035:2050]

In [ ]:
newmz[2035:2050]

In [ ]:
newmz[2035:2050] 

### Anchor point 3

In [ ]:
orgmz[5240:5250]

In [ ]:
newmz[5240:5250] # matches

In [ ]:
newmz[5240:5250] # matches

In [ ]:
import numpy as np
import msiwarp as mx

In [ ]:
# reference spectra
mzs_r = np.array(MALDIdataAnn2.var["m/z"]).astype(np.float) # peak m/z values
hs_r = np.mean(MALDIdataAnn2.X, axis = 0) # peak heights
s_r = [mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs_r, hs_r))]

In [ ]:
# unknown spectra
spectra = []
mzs = np.array(MALDIdataAnn1.var["m/z"]).astype(np.float) # peak m/z values
meanspectrum1 =  np.mean(MALDIdataAnn1.X, axis = 0)
for i in tqdm(range(0, MALDIdataAnn1.X.shape[0], 100)): # from 100 to 10 does not make a difference at anchor points
    hs = meanspectrum1
    spectra.append([mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs, hs))])

In [ ]:
len(spectra)

In [ ]:
### Setting up the warping nodes
node_mzs = [i for i in range(10,1030,102)]
node_deltas = [0.0001 * i for i in node_mzs] # slacks = node_deltas * n_steps
n_steps = 25
nodes = mx.initialize_nodes(node_mzs, node_deltas, n_steps)

epsilon = 1
n_cores = 4

In [ ]:
optimal_moves = mx.find_optimal_spectra_warpings(spectra, s_r, nodes, epsilon)

In [ ]:
spectra2 = [[mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs.astype(np.float), mzs))]]
warped_spectra = [mx.warp_peaks(s_i, nodes, o_i) for (s_i, o_i) in zip(spectra2, optimal_moves)]

In [ ]:
warped_spectra = [mx.warp_peaks(s_i, nodes, optimal_moves[0]) for s_i in spectra2]

In [ ]:
newmz = to_mz(warped_spectra[0])
oldmz = to_height(warped_spectra[0])

In [ ]:
newmz = []
newintensity = []

for s_i in zip(warped_spectra):
    newmz[i] = to_mz(s_i)
    newintensity[i] = to_height(s_i)

In [ ]:
len(newmz)

In [ ]:
oldmz

In [ ]:
warped_spectra = [mx.warp_peaks(s_i, nodes, o_i) for (s_i, o_i) in zip(s, optimal_moves)]

In [ ]:
MALDIdataAnn1.X

In [ ]:
meanspectrum_aligned = np.mean(MALDIdataAnn1.X, axis = 0)
meanspectrum_aligned_mv = np.convolve(meanspectrum_aligned, np.ones(3)/3, mode='valid')

### output the result

In [ ]:
mz_aligned = MALDIdataAnn1.var

In [ ]:
np.array(mz_aligned["m/z"][0:40])

In [ ]:
newmz[:-1]

In [ ]:
len(newmz)

In [ ]:
newmz_all = np.concatenate((np.array(mz_aligned["m/z"][1:40]), newmz[:-1]))

In [ ]:
len(newmz_all)

In [ ]:
newmz_all

In [ ]:
np.array(mz_aligned["m/z"][0:40])

In [ ]:
newmz_all[100:110]

In [ ]:
mz_aligned["m/z"][100:110]

In [ ]:
len(meanspectrum_aligned_mv)

In [ ]:
originaltable= pd.DataFrame(newmz_all)
originaltable = originaltable.rename(columns = {0:"mz_aligned_msi"})
originaltable["intensity_msi"] = meanspectrum_aligned_mv
originaltable

In [ ]:
csv_file_path = f'{OUTPUT_DIR}_Maaike'
originaltable.to_csv (csv_file_path + "/mean_spec_mv_msi.csv")